# Agent 第 08 课：Multi-Agent 协作

单个 agent 在复杂任务上会遇到一个问题：**system prompt 越堆越长、工具越加越多、风格要求越来越杂**——它同时要是程序员、研究员、编辑、审核员，分心又容易搞砸。

解决思路：**把一个大 agent 拆成多个小 agent**，每个只干一件事，彼此通过消息协作。

常见架构模式：

```
  A) Manager / Worker           （一个老板，多个下属）
                    ┌──> Worker_1 (搜索)
       Manager ─────┼──> Worker_2 (写作)
                    └──> Worker_3 (校对)

  B) Sequential Pipeline        （流水线）
       Researcher → Writer → Editor → Reviewer

  C) Debate / Consensus         （多个 agent 互相辩论，投票或合议）
       Agent_A ⇄ Agent_B ⇄ Agent_C  → Judge

  D) Blackboard                 （所有 agent 读写共享黑板）
       Agent_* ←→ shared state
```

这一课跑 **A 和 B 两种**——最常见、最好用、最容易理解。

In [ ]:
import json, re
from openai import OpenAI
client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
chat_model = next((m for m in ['qwen/qwen3.6-35b-a3b','qwen3.6-35b-a3b','qwen3.5-35b-a3b'] if m in model_ids), model_ids[0])
print('chat =', chat_model)

## 1. 模式 B：Sequential Pipeline（最简单）

每个 agent 是一个函数：`input -> output`。把它们串起来。

任务：给一个主题，写一篇 **150 字的技术短文 + 一段反驳 + 最终裁决**。

- `writer`：写支持观点
- `devil`：反驳
- `judge`：综合给出结论

In [ ]:
def call(system, user, temperature=0.3):
    r = client.chat.completions.create(model=chat_model, temperature=temperature,
        messages=[{'role':'system','content':system},{'role':'user','content':user}])
    return r.choices[0].message.content.strip()

WRITER_SYS = 'You are a technical writer. Write a concise (~150 words) argument in English supporting the given claim. Be specific, use concrete examples.'
DEVIL_SYS  = 'You are a skeptical engineer. Given a claim and a supporting argument, write a 100-word rebuttal that points out concrete weaknesses. Be specific.'
JUDGE_SYS  = 'You are a senior architect. Given a claim, a supporting argument, and a rebuttal, deliver a balanced verdict in 5-8 sentences: where the claim holds, where it does not, your recommendation.'

def pipeline(claim):
    support  = call(WRITER_SYS, f'Claim: {claim}')
    print('--- WRITER ---\n', support, '\n')
    rebuttal = call(DEVIL_SYS,  f'Claim: {claim}\n\nSupporting argument:\n{support}')
    print('--- DEVIL ---\n', rebuttal, '\n')
    verdict  = call(JUDGE_SYS,  f'Claim: {claim}\n\nSupport:\n{support}\n\nRebuttal:\n{rebuttal}')
    print('--- JUDGE ---\n', verdict)
    return {'support':support,'rebuttal':rebuttal,'verdict':verdict}

_ = pipeline('Rust should replace Python for ML training pipelines.')

## 2. 模式 A：Manager / Worker（带 routing）

Manager 看到用户请求，**自己不干活**，而是决定派给谁、给什么子任务。Worker 做完回交。Manager 汇总。

实现上，**Manager 也是一个带工具的 agent**——它的"工具"就是每个 Worker。

In [ ]:
# --- Workers：每个是一个 (system_prompt, fn) ---
def worker_researcher(task):
    sys = 'You are a researcher. Output 3-5 bullet points of relevant facts on the given topic. Use only well-known general knowledge; do not cite URLs.'
    return call(sys, task, temperature=0.2)

def worker_coder(task):
    sys = 'You are a Python coder. Write concise Python code (no explanation) that solves the task. Output code only, no fences.'
    code = call(sys, task, temperature=0)
    return re.sub(r'^```\w*\n|\n```$', '', code)

def worker_summarizer(task):
    sys = 'You summarize content into crisp, bulleted takeaways.'
    return call(sys, task, temperature=0.2)

WORKERS = {
    'researcher': ('Gather factual points on a topic.', worker_researcher),
    'coder':      ('Write a short Python snippet for a small task.', worker_coder),
    'summarizer': ('Condense text into key takeaways.', worker_summarizer),
}

In [ ]:
# --- Manager：用我们前几课的 Hermes 风格工具协议 ---
manager_tools = [
    {'name': name, 'description': desc,
     'parameters': {'type':'object','properties':{'task':{'type':'string'}},'required':['task']}}
    for name, (desc, _) in WORKERS.items()
] + [
    {'name':'finish','description':'Produce the final answer to the user when enough information has been gathered.',
     'parameters':{'type':'object','properties':{'answer':{'type':'string'}},'required':['answer']}}
]

MANAGER_SYSTEM = f"""You are a project manager coordinating specialist workers.

Workers (call them as tools):
{json.dumps(manager_tools, indent=2)}

Delegation rules:
- Break the user's request into sub-tasks, one worker per call.
- Use <tool_call>{{"name":"<worker>","arguments":{{"task":"<specific sub-task>"}}}}</tool_call>.
- When you have enough to answer, call `finish` with the final answer.
- Do NOT try to do a worker's job yourself."""

TOOL_CALL_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.S)

def manager_run(user_request, max_steps=6, verbose=True):
    messages = [{'role':'system','content':MANAGER_SYSTEM},{'role':'user','content':user_request}]
    for step in range(1, max_steps+1):
        r = client.chat.completions.create(model=chat_model, temperature=0, messages=messages)
        t = r.choices[0].message.content
        messages.append({'role':'assistant','content':t})
        if verbose: print(f'\n=== MANAGER step {step} ===\n{t}\n')
        m = TOOL_CALL_RE.search(t)
        if not m:
            messages.append({'role':'user','content':'Please issue exactly one <tool_call>.'}); continue
        call_obj = json.loads(m.group(1))
        name = call_obj['name']; args = call_obj.get('arguments') or {}
        if name == 'finish':
            return args.get('answer','')
        if name not in WORKERS:
            messages.append({'role':'user','content':f'<tool_response>{{"error":"unknown worker {name}"}}</tool_response>'}); continue
        _, fn = WORKERS[name]
        out = fn(args.get('task',''))
        if verbose: print(f'>>> worker[{name}] output ({len(out)} chars):\n{out[:300]}\n')
        messages.append({'role':'user','content':f'<tool_response>{json.dumps({"worker":name,"output":out})}</tool_response>'})
    return None

In [ ]:
ans = manager_run('Give me a 3-bullet primer on how vector databases work, plus a tiny Python snippet that computes cosine similarity between two numpy arrays.')
print('\n=========== FINAL ===========')
print(ans)

## 3. Multi-Agent 的陷阱

Multi-agent 看起来很炫，但要警惕：

### 陷阱 1：成本爆炸
单 agent 一次回答 3 次 LLM 调用；加 writer/devil/judge 就 3 倍；manager 带 3 个 worker，每个 worker 内部再是 agent——成本轻易 10 倍起步。

### 陷阱 2：沟通噪声
Agent 之间传的内容越多，每一跳都会有"理解偏差"。两三跳后可能偏离原任务。解决：**强制结构化 message schema**，每个 worker 的输出格式严格定义。

### 陷阱 3："假分工"
研究表明很多 multi-agent 系统**拆一个 agent 成三个 agent 后，效果还不如把原来单 agent 的 prompt 写好**。这叫"virtual division of labor"——看起来分工，其实只是把一个 LLM 的思考过程拆成三次调用。

**Anthropic 2024 的 paper 和很多团队的经验**：**先把单 agent 做到极致**（好的 system prompt、reflection、工具设计），还解决不了再上 multi-agent。

### 陷阱 4：死循环
Manager 派任务给 worker，worker 做不好，manager 再派，可能无限循环。必须有：
- `max_steps` 全局上限
- `per-worker retry limit`
- **可观测**：每次派发、每次返回都要有日志

## 4. 什么时候 Multi-Agent 真的值得

几个我认可的用例：

1. **不同步骤需要不同"人格/专业"的 system prompt**。写代码的 agent 和审核 PR 的 agent，角色冲突，合在一起会打架。拆开更稳。
2. **不同步骤的工具集差很远**。研究员要搜索、程序员要执行代码——合在一起工具太多 routing 容易错，拆开各自工具集小而精。
3. **需要并行**。5 个 worker 同时调研 5 个主题比一个 agent 串行快得多。Python 里 `asyncio` + `openai.AsyncClient` 轻松做到。
4. **不同步骤用不同模型**——manager 用贵模型出决策，worker 用便宜模型干活。

反过来，**不值得**的情况：
- 任务明显可以一个 prompt 答完
- 只是为了"架构好看"
- 业务还没稳定，加 multi-agent 会让调试难度上一个量级

## 5. 工程直觉

1. **Multi-agent 的本质就是"把 agent 当工具"**。回头看 manager 代码——worker 就是 tool，协议完全没变。这是为什么上一课的基础这么重要。
2. **接口先定**。每个 agent 的输入/输出 schema 写清楚（pydantic），不要让下一个 agent 去解析散文。
3. **默认从 Sequential Pipeline 起步**。Manager/Worker 灵活但开销大；流水线足够解决 80% 问题。
4. **日志 + trace 是 multi-agent 的救命稻草**。每条消息打 `agent_id`、`step_id`、`parent_step_id`，用 mermaid/graphviz 画出来。
5. **先怀疑单 agent 做不好**。加一个 worker 前，问自己：把 system prompt 重写、加 reflection、换强模型，能不能解决？大多数时候能。

---

下一课：**第 09 课 Agent 评测**——怎么回答"这个 agent 到底好不好"？任务成功率、步数、token 消耗、失败模式分析，全都得量化。